# Trabalho CBA 

In [74]:
import pandas as pd
import optuna
import xgboost as xgb
import numpy as np
from sklearn.metrics import matthews_corrcoef
from sklearn.model_selection import StratifiedGroupKFold
from joblib import Parallel, delayed
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_contour

import pandas as pd
import numpy as np
import os
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, 
    f1_score, matthews_corrcoef, roc_auc_score, confusion_matrix
)

## Separação dos Dados

In [57]:
# Importar conjunto de dados

dados = pd.read_parquet("Batimentos_Com_Estatisticas.parquet")

registros = dados["Registro"]
tipos = dados["Tipo"]

print(dados.shape)
print(dados.columns)

(89583, 318)
Index(['Registro', 'Tipo', 'Atributo_1', 'Atributo_2', 'Atributo_3',
       'Atributo_4', 'Atributo_5', 'Atributo_6', 'Atributo_7', 'Atributo_8',
       ...
       'desvio_padrao_pre_r', 'soma_quadratica_pre_r', 'norma_pre_r',
       'rms_pre_r', 'media_post_r', 'desvio_padrao_post_r',
       'soma_quadratica_post_r', 'norma_post_r', 'rms_post_r', 'Padrao_AAMI'],
      dtype='object', length=318)


In [58]:
# Binarizar saída
y = dados['Padrao_AAMI']

y_bin = pd.Series([0 if letra == "N" else 1 for letra in y], name='Padrao_AAMI', index=dados.index)

In [59]:
# Separar features que serão usadas em cada modelo (9)

# 1 - Apenas o sinal
sinal = dados[[f'Atributo_{i}' for i in range(1,301)]]

# 2 - Estatisticas Globais
estat_globais = dados[['media_global', 'desvio_padrao_global',
        'soma_quadratica_global', 'norma_global', 'rms_global']]

estat_pre_r = dados[['media_pre_r', 'desvio_padrao_pre_r',
       'soma_quadratica_pre_r', 'norma_pre_r', 'rms_pre_r']]

estat_post_r = dados[['media_post_r', 'desvio_padrao_post_r',
       'soma_quadratica_post_r', 'norma_post_r', 'rms_post_r']]

# 3- Estatística global + estatística antes do pico R
estat_globais_pre_r = pd.concat([estat_globais, estat_pre_r], axis=1)

# 4 - Estatística global + estatística depois do pico R
estat_globais_post_r = pd.concat([estat_globais, estat_post_r], axis=1)

# 5 - Estatística global + antes + depois
estat_globais_pre_post_r = pd.concat([estat_globais, estat_pre_r, estat_post_r], axis=1)

# 6 - Sinal + estatística global
sinal_estat_global = pd.concat([sinal, estat_globais], axis=1)

# 7 - Sinal + estatística global + antes
sinal_estat_global_pre = pd.concat([sinal, estat_globais, estat_pre_r], axis=1)

# 8 - Sinal + estatística global + depois
sinal_estat_global_post = pd.concat([sinal, estat_globais, estat_post_r], axis=1)

# 9 - Sinal + estatística global + antes + depois
sinal_estat_global_pre_post = pd.concat([sinal, estat_globais, estat_pre_r, estat_post_r], axis=1)

# Imprimir quantidade de colunas de cada variável
print(f"sinal: {sinal.shape[1]} colunas")
print(f"estat_globais: {estat_globais.shape[1]} colunas")
print(f"estat_pre_r: {estat_pre_r.shape[1]} colunas")
print(f"estat_post_r: {estat_post_r.shape[1]} colunas")
print(f"estat_globais_pre_r: {estat_globais_pre_r.shape[1]} colunas")
print(f"estat_globais_post_r: {estat_globais_post_r.shape[1]} colunas")
print(f"estat_globais_pre_post_r: {estat_globais_pre_post_r.shape[1]} colunas")
print(f"sinal_estat_global: {sinal_estat_global.shape[1]} colunas")
print(f"sinal_estat_global_pre: {sinal_estat_global_pre.shape[1]} colunas")
print(f"sinal_estat_global_post: {sinal_estat_global_post.shape[1]} colunas")
print(f"sinal_estat_global_pre_post: {sinal_estat_global_pre_post.shape[1]} colunas")

sinal: 300 colunas
estat_globais: 5 colunas
estat_pre_r: 5 colunas
estat_post_r: 5 colunas
estat_globais_pre_r: 10 colunas
estat_globais_post_r: 10 colunas
estat_globais_pre_post_r: 15 colunas
sinal_estat_global: 305 colunas
sinal_estat_global_pre: 310 colunas
sinal_estat_global_post: 310 colunas
sinal_estat_global_pre_post: 315 colunas


In [60]:
# Separar em teste e treino
is_treino = tipos == 'Treino'
is_teste = tipos == 'Teste'

# Registros
registros_treino = registros.loc[is_treino].copy()
registros_teste = registros.loc[is_teste].copy()

# Labels
y_treino = y_bin.loc[is_treino].copy()
y_teste = y_bin.loc[is_teste].copy()

# Features
# 1 - Sinal
X_treino_sinal = sinal.loc[is_treino].copy()
X_teste_sinal = sinal.loc[is_teste].copy()

# 2 - Estatisticas Globais
X_treino_estat_globais = estat_globais.loc[is_treino].copy()
X_teste_estat_globais = estat_globais.loc[is_teste].copy()

# 3 - Estatística global + estatística antes do pico R
X_treino_estat_globais_pre_r = estat_globais_pre_r.loc[is_treino].copy()
X_teste_estat_globais_pre_r = estat_globais_pre_r.loc[is_teste].copy()

# 4 - Estatística global + estatística depois do pico R
X_treino_estat_globais_post_r = estat_globais_post_r.loc[is_treino].copy()
X_teste_estat_globais_post_r = estat_globais_post_r.loc[is_teste].copy()

# 5 - Estatística global + antes + depois
X_treino_estat_globais_pre_post_r = estat_globais_pre_post_r.loc[is_treino].copy()
X_teste_estat_globais_pre_post_r = estat_globais_pre_post_r.loc[is_teste].copy()

# 6 - Sinal + estatística global
X_treino_sinal_estat_global = sinal_estat_global.loc[is_treino].copy()
X_teste_sinal_estat_global = sinal_estat_global.loc[is_teste].copy()

# 7 - Sinal + estatística global + antes
X_treino_sinal_estat_global_pre = sinal_estat_global_pre.loc[is_treino].copy()
X_teste_sinal_estat_global_pre = sinal_estat_global_pre.loc[is_teste].copy()

# 8 - Sinal + estatística global + depois
X_treino_sinal_estat_global_post = sinal_estat_global_post.loc[is_treino].copy()
X_teste_sinal_estat_global_post = sinal_estat_global_post.loc[is_teste].copy()

# 9 - Sinal + estatística global + antes + depois
X_treino_sinal_estat_global_pre_post = sinal_estat_global_pre_post.loc[is_treino].copy()
X_teste_sinal_estat_global_pre_post = sinal_estat_global_pre_post.loc[is_teste].copy()

## Treinamento dos Modelos

In [81]:
# Peso na classe positiva

num_1_treino = sum(1 for valor in y_treino if valor == 1)  # Arritmias
num_0_treino = sum(1 for valor in y_treino if valor == 0)  # Normal

scale_pos_weight = (2 * num_0_treino / num_1_treino) if num_1_treino > 0 else 1

print(scale_pos_weight)

31.47177848775293


### Funções

In [82]:
def objective(trial, X_treino, y_treino):
    param = {
        "learning_rate": trial.suggest_float("learning_rate", 0.0001, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 350),
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "scale_pos_weight": scale_pos_weight,
        "random_state": 14,
        "n_jobs": 1, 
        "eval_metric": "auc"
    }

    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=14)
    
    def train_fold(train_index, val_index):
        X_fold_treino = X_treino.iloc[train_index]
        X_fold_val = X_treino.iloc[val_index]
        y_fold_treino = y_treino.iloc[train_index]
        y_fold_val = y_treino.iloc[val_index]

        model = xgb.XGBClassifier(**param)

        model.fit(X_fold_treino, y_fold_treino,)

        preds = model.predict(X_fold_val)
        return matthews_corrcoef(y_fold_val, preds)

    # Paralelizar os folds com joblib
    folds = list(sgkf.split(X_treino, y_treino, groups=registros_treino))
    scores = Parallel(n_jobs=-1, backend="loky")(
        delayed(train_fold)(train_idx, val_idx) for train_idx, val_idx in folds
    )

    return np.mean(scores)

In [91]:
def atualizar_excel_metricas(arquivo, cenario_id, descricao, y_true, y_pred, y_proba):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    especificidade = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    
    nova_linha = {
        'Cenário': cenario_id,
        'Descrição': descricao,
        'Acurácia': accuracy_score(y_true, y_pred),
        'Sensibilidade': recall_score(y_true, y_pred),
        'Especificidade': especificidade,
        'Precisão': precision_score(y_true, y_pred, zero_division=0),
        'F1 Score': f1_score(y_true, y_pred),
        'MCC': matthews_corrcoef(y_true, y_pred), 
        'AUC ROC': roc_auc_score(y_true, y_proba)
    }
    
    df_novo = pd.DataFrame([nova_linha])
    
    if os.path.exists(arquivo):
        df_antigo = pd.read_excel(arquivo)
        df_antigo = df_antigo[df_antigo['Cenário'] != cenario_id]
        df_final = pd.concat([df_antigo, df_novo], ignore_index=True).sort_values('Cenário')
    else:
        df_final = df_novo
        
    df_final.to_excel(arquivo, index=False)
    print(f"Métricas do Cenário {cenario_id} salvas em: {arquivo}")

In [84]:
def atualizar_parquet_predicoes(arquivo, cenario_id, y_true, y_pred, y_proba):
    coluna_prob = f'prob_c{cenario_id}'
    coluna_pred = f'pred_c{cenario_id}'
    
    if os.path.exists(arquivo):
        df = pd.read_parquet(arquivo)
        df[coluna_prob] = y_proba
    else:
        df = pd.DataFrame({
            'y_true': y_true,
            coluna_prob: y_proba,
            coluna_pred: y_pred
        })
        
    df.to_parquet(arquivo, index=False)
    print(f"Probabilidades do Cenário {cenario_id} salvas em: {arquivo}")

### 1 - Modelo para o Sinal

In [103]:
study_1 = optuna.create_study(direction="maximize")
study_1.optimize(lambda trial: objective(trial, X_treino_sinal, y_treino), n_trials=15, show_progress_bar=True)

plot_optimization_history(study_1).show()
plot_param_importances(study_1).show()
plot_contour(study_1).show()

[I 2026-05-01 13:43:38,730] A new study created in memory with name: no-name-5c5a898e-176d-4f1d-a597-b74099f589e8
Best trial: 0. Best value: 0:   7%|▋         | 1/15 [00:14<03:17, 14.09s/it]

[I 2026-05-01 13:43:52,821] Trial 0 finished with value: 0.0 and parameters: {'learning_rate': 0.00012325278371173312, 'max_depth': 3, 'n_estimators': 149}. Best is trial 0 with value: 0.0.


Best trial: 1. Best value: 0.143426:  13%|█▎        | 2/15 [05:46<43:36, 201.25s/it]

[I 2026-05-01 13:49:25,084] Trial 1 finished with value: 0.14342563310639206 and parameters: {'learning_rate': 0.0014359717716053303, 'max_depth': 296, 'n_estimators': 255}. Best is trial 1 with value: 0.14342563310639206.


Best trial: 1. Best value: 0.143426:  20%|██        | 3/15 [11:45<54:39, 273.29s/it]

...

[I 2026-05-01 14:24:50,815] Trial 12 finished with value: 0.28645458293390724 and parameters: {'learning_rate': 0.0695839682544292, 'max_depth': 174, 'n_estimators': 112}. Best is trial 7 with value: 0.2935657974519263.


Best trial: 7. Best value: 0.293566:  93%|█████████▎| 14/15 [43:07<01:57, 117.94s/it]

[I 2026-05-01 14:26:46,492] Trial 13 finished with value: 0.2867517950544928 and parameters: {'learning_rate': 0.05571689793894453, 'max_depth': 257, 'n_estimators': 116}. Best is trial 7 with value: 0.2935657974519263.


Best trial: 7. Best value: 0.293566: 100%|██████████| 15/15 [44:55<00:00, 179.72s/it]

[I 2026-05-01 14:28:34,496] Trial 14 finished with value: 0.291210735227947 and parameters: {'learning_rate': 0.06391036255716875, 'max_depth': 275, 'n_estimators': 117}. Best is trial 7 with value: 0.2935657974519263.


In [104]:
melhores_params_1 = study_1.best_params

modelo_1 = xgb.XGBClassifier(**melhores_params_1)
modelo_1.fit(X_treino_sinal, y_treino)

pred_1 = modelo_1.predict(X_teste_sinal)
prob_1 = modelo_1.predict_proba(X_teste_sinal)[:, 1]

atualizar_excel_metricas(
    arquivo="resultados_cenarios.xlsx",
    cenario_id=1,
    descricao="Sinal",
    y_true=y_teste,
    y_pred=pred_1,
    y_proba=prob_1
)

atualizar_parquet_predicoes(
    arquivo="predicoes_detalhadas.parquet",
    cenario_id=1,
    y_true=y_teste,
    y_pred=pred_1,
    y_proba=prob_1
)

Métricas do Cenário 1 salvas em: resultados_cenarios.xlsx
Probabilidades do Cenário 1 salvas em: predicoes_detalhadas.parquet


### 2 - Modelo para Estisticas Globais

In [ ]:
study_2 = optuna.create_study(direction="maximize")
study_2.optimize(lambda trial: objective(trial, X_treino_estat_globais, y_treino), n_trials=50, show_progress_bar=True)

plot_optimization_history(study_2).show()
plot_param_importances(study_2).show()
plot_contour(study_2).show()

[I 2026-05-01 11:18:31,470] A new study created in memory with name: no-name-0279b277-1a19-4262-a52f-7d9228a3e752
Best trial: 0. Best value: 0.221418:   2%|▏         | 1/50 [00:40<32:53, 40.27s/it]

[I 2026-05-01 11:19:11,746] Trial 0 finished with value: 0.22141823166183544 and parameters: {'learning_rate': 0.008097226945625563, 'max_depth': 153, 'n_estimators': 233}. Best is trial 0 with value: 0.22141823166183544.


Best trial: 0. Best value: 0.221418:   4%|▍         | 2/50 [01:06<25:23, 31.73s/it]

[I 2026-05-01 11:19:37,500] Trial 1 finished with value: 0.1906548896909818 and parameters: {'learning_rate': 0.0882297604053781, 'max_depth': 97, 'n_estimators': 247}. Best is trial 0 with value: 0.22141823166183544.


Best trial: 0. Best value: 0.221418:   6%|▌         | 3/50 [01:23<19:45, 25.22s/it]

...

[I 2026-05-01 11:39:48,323] Trial 47 finished with value: 0.18406800140214324 and parameters: {'learning_rate': 0.11889508494272612, 'max_depth': 244, 'n_estimators': 245}. Best is trial 44 with value: 0.23071730833199763.


Best trial: 44. Best value: 0.230717:  98%|█████████▊| 49/50 [21:46<00:29, 29.84s/it]

[I 2026-05-01 11:40:17,548] Trial 48 finished with value: 0.20999447399006343 and parameters: {'learning_rate': 0.03756039357941608, 'max_depth': 274, 'n_estimators': 230}. Best is trial 44 with value: 0.23071730833199763.


Best trial: 44. Best value: 0.230717: 100%|██████████| 50/50 [22:12<00:00, 26.65s/it]


[I 2026-05-01 11:40:43,736] Trial 49 finished with value: 0.0 and parameters: {'learning_rate': 0.00010519739040270379, 'max_depth': 321, 'n_estimators': 264}. Best is trial 44 with value: 0.23071730833199763.


In [ ]:
melhores_params_2 = study_2.best_params

modelo_2 = xgb.XGBClassifier(**melhores_params_2)
modelo_2.fit(X_treino_estat_globais, y_treino)

pred_2 = modelo_2.predict(X_teste_estat_globais)
prob_2 = modelo_2.predict_proba(X_teste_estat_globais)[:, 1]

atualizar_excel_metricas(
    arquivo="resultados_cenarios.xlsx",
    cenario_id=2,
    descricao="Estatísticas Globais",
    y_true=y_teste,
    y_pred=pred_2,
    y_proba=prob_2
)

atualizar_parquet_predicoes(
    arquivo="predicoes_detalhadas.parquet",
    cenario_id=2,
    y_true=y_teste,
    y_pred=pred_2,
    y_proba=prob_2
)

Métricas do Cenário 2 salvas em: resultados_cenarios.xlsx
Probabilidades do Cenário 2 salvas em: predicoes_detalhadas.parquet


### 3 - Estatística global + estatística antes do pico R

In [87]:
study_3 = optuna.create_study(direction="maximize")
study_3.optimize(lambda trial: objective(trial, X_treino_estat_globais_pre_r, y_treino), n_trials=50, show_progress_bar=True)

plot_optimization_history(study_3).show()
plot_param_importances(study_3).show()
plot_contour(study_3).show()

[I 2026-05-01 12:10:31,781] A new study created in memory with name: no-name-996bdb29-07f0-4d44-bf95-8ec32878afc6
Best trial: 0. Best value: 0:   2%|▏         | 1/50 [00:36<29:57, 36.69s/it]

[I 2026-05-01 12:11:08,478] Trial 0 finished with value: 0.0 and parameters: {'learning_rate': 0.00036742892129919746, 'max_depth': 180, 'n_estimators': 289}. Best is trial 0 with value: 0.0.


Best trial: 1. Best value: 0.200554:   4%|▍         | 2/50 [00:45<16:23, 20.48s/it]

[I 2026-05-01 12:11:17,613] Trial 1 finished with value: 0.20055426695183737 and parameters: {'learning_rate': 0.24858669745181064, 'max_depth': 299, 'n_estimators': 150}. Best is trial 1 with value: 0.20055426695183737.


Best trial: 2. Best value: 0.201689:   6%|▌         | 3/50 [01:01<14:22, 18.35s/it]

...

[I 2026-05-01 12:26:58,965] Trial 47 finished with value: 0.19463426654014163 and parameters: {'learning_rate': 0.23808687980333548, 'max_depth': 126, 'n_estimators': 187}. Best is trial 38 with value: 0.2075018574593524.


Best trial: 38. Best value: 0.207502:  98%|█████████▊| 49/50 [16:58<00:23, 23.60s/it]

[I 2026-05-01 12:27:30,636] Trial 48 finished with value: 0.2047530338577101 and parameters: {'learning_rate': 0.012404967669225454, 'max_depth': 157, 'n_estimators': 230}. Best is trial 38 with value: 0.2075018574593524.


Best trial: 38. Best value: 0.207502: 100%|██████████| 50/50 [17:33<00:00, 21.07s/it]

[I 2026-05-01 12:28:05,506] Trial 49 finished with value: 0.19537635377147874 and parameters: {'learning_rate': 0.006258256836771008, 'max_depth': 197, 'n_estimators': 251}. Best is trial 38 with value: 0.2075018574593524.


In [96]:
melhores_params_3 = study_3.best_params

modelo_3 = xgb.XGBClassifier(**melhores_params_3)
modelo_3.fit(X_treino_estat_globais_pre_r, y_treino)

pred_3 = modelo_3.predict(X_teste_estat_globais_pre_r)
prob_3 = modelo_3.predict_proba(X_teste_estat_globais_pre_r)[:, 1]

atualizar_excel_metricas(
    arquivo="resultados_cenarios.xlsx",
    cenario_id=3,
    descricao="Estatísticas Globais + Antes do Pico R",
    y_true=y_teste,
    y_pred=pred_3,
    y_proba=prob_3
)

atualizar_parquet_predicoes(
    arquivo="predicoes_detalhadas.parquet",
    cenario_id=3,
    y_true=y_teste,
    y_pred=pred_3,
    y_proba=prob_3
)

Métricas do Cenário 3 salvas em: resultados_cenarios.xlsx
Probabilidades do Cenário 3 salvas em: predicoes_detalhadas.parquet


### 4 - Estatística global + estatística depois do pico R

In [94]:
study_4= optuna.create_study(direction="maximize")
study_4.optimize(lambda trial: objective(trial, X_treino_estat_globais_post_r, y_treino), n_trials=50, show_progress_bar=True)

plot_optimization_history(study_4).show()
plot_param_importances(study_4).show()
plot_contour(study_4).show()

[I 2026-05-01 12:45:58,671] A new study created in memory with name: no-name-8b6686f8-ee7c-4c0b-8d69-db84ba6b8c73
Best trial: 0. Best value: 0:   2%|▏         | 1/50 [00:04<03:50,  4.69s/it]

[I 2026-05-01 12:46:03,370] Trial 0 finished with value: 0.0 and parameters: {'learning_rate': 0.00012622279840205955, 'max_depth': 6, 'n_estimators': 219}. Best is trial 0 with value: 0.0.


Best trial: 1. Best value: 0.195522:   4%|▍         | 2/50 [00:40<18:27, 23.06s/it]

[I 2026-05-01 12:46:39,290] Trial 1 finished with value: 0.19552166935317405 and parameters: {'learning_rate': 0.008598250099552382, 'max_depth': 28, 'n_estimators': 223}. Best is trial 1 with value: 0.19552166935317405.


Best trial: 1. Best value: 0.195522:   6%|▌         | 3/50 [01:21<24:20, 31.07s/it]

...

[I 2026-05-01 12:57:39,067] Trial 47 finished with value: 0.2015977901537193 and parameters: {'learning_rate': 0.06428707607142024, 'max_depth': 117, 'n_estimators': 108}. Best is trial 3 with value: 0.2040264361002023.


Best trial: 3. Best value: 0.204026:  98%|█████████▊| 49/50 [11:57<00:13, 13.32s/it]

[I 2026-05-01 12:57:56,490] Trial 48 finished with value: 0.20145577870196835 and parameters: {'learning_rate': 0.04489928610446426, 'max_depth': 128, 'n_estimators': 111}. Best is trial 3 with value: 0.2040264361002023.


Best trial: 3. Best value: 0.204026: 100%|██████████| 50/50 [12:08<00:00, 14.58s/it]

[I 2026-05-01 12:58:07,508] Trial 49 finished with value: 0.19742693959747853 and parameters: {'learning_rate': 0.07064391491274015, 'max_depth': 103, 'n_estimators': 57}. Best is trial 3 with value: 0.2040264361002023.


In [98]:
melhores_params_4 = study_4.best_params

modelo_4 = xgb.XGBClassifier(**melhores_params_4)
modelo_4.fit(X_treino_estat_globais_post_r, y_treino)

pred_4 = modelo_4.predict(X_teste_estat_globais_post_r)
prob_4 = modelo_4.predict_proba(X_teste_estat_globais_post_r)[:, 1]

atualizar_excel_metricas(
    arquivo="resultados_cenarios.xlsx",
    cenario_id=4,
    descricao="Estatísticas Globais + Depois do Pico R",
    y_true=y_teste,
    y_pred=pred_4,
    y_proba=prob_4
)

atualizar_parquet_predicoes(
    arquivo="predicoes_detalhadas.parquet",
    cenario_id=4,
    y_true=y_teste,
    y_pred=pred_4,
    y_proba=prob_4
)

Métricas do Cenário 4 salvas em: resultados_cenarios.xlsx
Probabilidades do Cenário 4 salvas em: predicoes_detalhadas.parquet


### 5 - Estatística global + Antes + Depois

In [99]:
study_5= optuna.create_study(direction="maximize")
study_5.optimize(lambda trial: objective(trial, X_treino_estat_globais_pre_post_r, y_treino), n_trials=50, show_progress_bar=True)

plot_optimization_history(study_5).show()
plot_param_importances(study_5).show()
plot_contour(study_5).show()

[I 2026-05-01 13:07:57,913] A new study created in memory with name: no-name-f939705c-6dd5-43d8-8262-12f82c94f42d
Best trial: 0. Best value: 0.189987:   2%|▏         | 1/50 [00:41<33:36, 41.16s/it]

[I 2026-05-01 13:08:39,079] Trial 0 finished with value: 0.18998704622399204 and parameters: {'learning_rate': 0.00514394045539915, 'max_depth': 259, 'n_estimators': 203}. Best is trial 0 with value: 0.18998704622399204.


Best trial: 0. Best value: 0.189987:   4%|▍         | 2/50 [01:16<30:16, 37.85s/it]

[I 2026-05-01 13:09:14,613] Trial 1 finished with value: 0.0 and parameters: {'learning_rate': 0.001048528610391685, 'max_depth': 348, 'n_estimators': 198}. Best is trial 0 with value: 0.18998704622399204.


Best trial: 0. Best value: 0.189987:   6%|▌         | 3/50 [01:55<30:00, 38.31s/it]

...

[I 2026-05-01 13:27:43,910] Trial 47 finished with value: 0.1881472146913884 and parameters: {'learning_rate': 0.006608921311304594, 'max_depth': 319, 'n_estimators': 152}. Best is trial 14 with value: 0.20979182722896234.


Best trial: 14. Best value: 0.209792:  98%|█████████▊| 49/50 [20:26<00:32, 32.13s/it]

[I 2026-05-01 13:28:24,132] Trial 48 finished with value: 0.19676999556789962 and parameters: {'learning_rate': 0.01580905805400378, 'max_depth': 290, 'n_estimators': 224}. Best is trial 14 with value: 0.20979182722896234.


Best trial: 14. Best value: 0.209792: 100%|██████████| 50/50 [20:46<00:00, 24.93s/it]

[I 2026-05-01 13:28:44,207] Trial 49 finished with value: 0.0 and parameters: {'learning_rate': 0.0021499634249520817, 'max_depth': 114, 'n_estimators': 107}. Best is trial 14 with value: 0.20979182722896234.


In [100]:
melhores_params_5 = study_5.best_params

modelo_5 = xgb.XGBClassifier(**melhores_params_5)
modelo_5.fit(X_treino_estat_globais_pre_post_r, y_treino)

pred_5 = modelo_5.predict(X_teste_estat_globais_pre_post_r)
prob_5 = modelo_5.predict_proba(X_teste_estat_globais_pre_post_r)[:, 1]

atualizar_excel_metricas(
    arquivo="resultados_cenarios.xlsx",
    cenario_id=5,
    descricao="Estatísticas Globais + Antes + Depois do Pico R",
    y_true=y_teste,
    y_pred=pred_5,
    y_proba=prob_5
)

atualizar_parquet_predicoes(
    arquivo="predicoes_detalhadas.parquet",
    cenario_id=5,
    y_true=y_teste,
    y_pred=pred_5,
    y_proba=prob_5
)

Métricas do Cenário 5 salvas em: resultados_cenarios.xlsx
Probabilidades do Cenário 5 salvas em: predicoes_detalhadas.parquet


### 6 - Sinal + estatística global

In [106]:
study_6= optuna.create_study(direction="maximize")
study_6.optimize(lambda trial: objective(trial, X_treino_sinal_estat_global, y_treino), n_trials=15, show_progress_bar=True)

plot_optimization_history(study_6).show()
plot_param_importances(study_6).show()
plot_contour(study_6).show()

[I 2026-05-01 14:32:58,578] A new study created in memory with name: no-name-12548823-cb78-4a0a-95bf-29b62d26ea33
Best trial: 0. Best value: 0.323306:   7%|▋         | 1/15 [03:15<45:35, 195.37s/it]

[I 2026-05-01 14:36:13,939] Trial 0 finished with value: 0.32330636691847936 and parameters: {'learning_rate': 0.02498146520044292, 'max_depth': 319, 'n_estimators': 130}. Best is trial 0 with value: 0.32330636691847936.


Best trial: 1. Best value: 0.349393:  13%|█▎        | 2/15 [04:48<29:19, 135.37s/it]

[I 2026-05-01 14:37:47,311] Trial 1 finished with value: 0.34939304117896547 and parameters: {'learning_rate': 0.17384302658819392, 'max_depth': 229, 'n_estimators': 266}. Best is trial 1 with value: 0.34939304117896547.


Best trial: 1. Best value: 0.349393:  20%|██        | 3/15 [06:20<23:07, 115.67s/it]

...

[I 2026-05-01 15:09:50,393] Trial 12 finished with value: 0.3439374457826134 and parameters: {'learning_rate': 0.15306507586152515, 'max_depth': 206, 'n_estimators': 245}. Best is trial 1 with value: 0.34939304117896547.


Best trial: 13. Best value: 0.356878:  93%|█████████▎| 14/15 [39:14<02:01, 121.23s/it]

[I 2026-05-01 15:12:13,392] Trial 13 finished with value: 0.3568781173856016 and parameters: {'learning_rate': 0.06519921618357065, 'max_depth': 107, 'n_estimators': 237}. Best is trial 13 with value: 0.3568781173856016.


Best trial: 13. Best value: 0.356878: 100%|██████████| 15/15 [40:44<00:00, 162.99s/it]

[I 2026-05-01 15:13:43,482] Trial 14 finished with value: 0.3188940341330485 and parameters: {'learning_rate': 0.056701739438293944, 'max_depth': 9, 'n_estimators': 235}. Best is trial 13 with value: 0.3568781173856016.


In [108]:
melhores_params_6 = study_6.best_params

modelo_6 = xgb.XGBClassifier(**melhores_params_6)
modelo_6.fit(X_treino_sinal_estat_global, y_treino)

pred_6 = modelo_6.predict(X_teste_sinal_estat_global)
prob_6 = modelo_6.predict_proba(X_teste_sinal_estat_global)[:, 1]

atualizar_excel_metricas(
    arquivo="resultados_cenarios.xlsx",
    cenario_id=6,
    descricao="Sinal + Estatísticas Globais",
    y_true=y_teste,
    y_pred=pred_6,
    y_proba=prob_6
)

atualizar_parquet_predicoes(
    arquivo="predicoes_detalhadas.parquet",
    cenario_id=6,
    y_true=y_teste,
    y_pred=pred_6,
    y_proba=prob_6
)

Métricas do Cenário 6 salvas em: resultados_cenarios.xlsx
Probabilidades do Cenário 6 salvas em: predicoes_detalhadas.parquet


### 7 - Sinal + estatística global + antes

In [109]:
study_7= optuna.create_study(direction="maximize")
study_7.optimize(lambda trial: objective(trial, X_treino_sinal_estat_global_pre, y_treino), n_trials=15, show_progress_bar=True)

plot_optimization_history(study_7).show()
plot_param_importances(study_7).show()
plot_contour(study_7).show()

[I 2026-05-01 15:17:17,983] A new study created in memory with name: no-name-4c4111aa-7436-4d46-b4e6-5078b729d3fd
Best trial: 0. Best value: 0.328636:   7%|▋         | 1/15 [04:36<1:04:33, 276.65s/it]

[I 2026-05-01 15:21:54,621] Trial 0 finished with value: 0.32863614188162593 and parameters: {'learning_rate': 0.004910772231006597, 'max_depth': 67, 'n_estimators': 190}. Best is trial 0 with value: 0.32863614188162593.


Best trial: 0. Best value: 0.328636:  13%|█▎        | 2/15 [10:45<1:11:40, 330.80s/it]

[I 2026-05-01 15:28:03,314] Trial 1 finished with value: 0.0 and parameters: {'learning_rate': 0.0005667125756317057, 'max_depth': 95, 'n_estimators': 255}. Best is trial 0 with value: 0.32863614188162593.


Best trial: 2. Best value: 0.349565:  20%|██        | 3/15 [12:35<45:59, 229.92s/it]  

...

[I 2026-05-01 15:56:46,662] Trial 12 finished with value: 0.36476620831032525 and parameters: {'learning_rate': 0.04273502366954877, 'max_depth': 220, 'n_estimators': 142}. Best is trial 12 with value: 0.36476620831032525.


Best trial: 13. Best value: 0.369464:  93%|█████████▎| 14/15 [42:56<02:35, 155.40s/it]

[I 2026-05-01 16:00:14,780] Trial 13 finished with value: 0.3694641441911036 and parameters: {'learning_rate': 0.03395833025711918, 'max_depth': 194, 'n_estimators': 209}. Best is trial 13 with value: 0.3694641441911036.


Best trial: 13. Best value: 0.369464: 100%|██████████| 15/15 [47:04<00:00, 188.30s/it]

[I 2026-05-01 16:04:22,529] Trial 14 finished with value: 0.36934703850201434 and parameters: {'learning_rate': 0.02514275675878572, 'max_depth': 337, 'n_estimators': 222}. Best is trial 13 with value: 0.3694641441911036.


In [111]:
melhores_params_7 = study_7.best_params

modelo_7 = xgb.XGBClassifier(**melhores_params_7)
modelo_7.fit(X_treino_sinal_estat_global_pre, y_treino)

pred_7 = modelo_7.predict(X_teste_sinal_estat_global_pre)
prob_7 = modelo_7.predict_proba(X_teste_sinal_estat_global_pre)[:, 1]

atualizar_excel_metricas(
    arquivo="resultados_cenarios.xlsx",
    cenario_id=7,
    descricao="Sinal + Estatísticas Globais e Antes",
    y_true=y_teste,
    y_pred=pred_7,
    y_proba=prob_7
)

atualizar_parquet_predicoes(
    arquivo="predicoes_detalhadas.parquet",
    cenario_id=7,
    y_true=y_teste,
    y_pred=pred_7,
    y_proba=prob_7
)

Métricas do Cenário 7 salvas em: resultados_cenarios.xlsx
Probabilidades do Cenário 7 salvas em: predicoes_detalhadas.parquet


### 8 - Sinal + estatística global + depois

In [112]:
study_8= optuna.create_study(direction="maximize")
study_8.optimize(lambda trial: objective(trial, X_treino_sinal_estat_global_post, y_treino), n_trials=15, show_progress_bar=True)

plot_optimization_history(study_8).show()
plot_param_importances(study_8).show()
plot_contour(study_8).show()

[I 2026-05-01 16:08:02,347] A new study created in memory with name: no-name-7e446851-0fef-4e71-8938-44f82132cf39
Best trial: 0. Best value: 0:   7%|▋         | 1/15 [06:11<1:26:36, 371.17s/it]

[I 2026-05-01 16:14:13,505] Trial 0 finished with value: 0.0 and parameters: {'learning_rate': 0.00018585076957796965, 'max_depth': 164, 'n_estimators': 257}. Best is trial 0 with value: 0.0.


Best trial: 0. Best value: 0:  13%|█▎        | 2/15 [08:33<51:13, 236.45s/it]  

[I 2026-05-01 16:16:35,659] Trial 1 finished with value: 0.0 and parameters: {'learning_rate': 0.0003656391761764389, 'max_depth': 168, 'n_estimators': 89}. Best is trial 0 with value: 0.0.


Best trial: 2. Best value: 0.348536:  20%|██        | 3/15 [10:00<33:39, 168.33s/it]

...

[I 2026-05-01 16:54:58,326] Trial 12 finished with value: 0.37818395256491466 and parameters: {'learning_rate': 0.024439039707818046, 'max_depth': 251, 'n_estimators': 299}. Best is trial 11 with value: 0.387411882103868.


Best trial: 11. Best value: 0.387412:  93%|█████████▎| 14/15 [48:29<03:15, 195.53s/it]

[I 2026-05-01 16:56:32,323] Trial 13 finished with value: 0.2832102367029627 and parameters: {'learning_rate': 0.030532325030196223, 'max_depth': 235, 'n_estimators': 55}. Best is trial 11 with value: 0.387411882103868.


Best trial: 11. Best value: 0.387412: 100%|██████████| 15/15 [53:00<00:00, 212.01s/it]

[I 2026-05-01 17:01:02,429] Trial 14 finished with value: 0.34837669736466104 and parameters: {'learning_rate': 0.01867559012774517, 'max_depth': 340, 'n_estimators': 213}. Best is trial 11 with value: 0.387411882103868.


In [113]:
melhores_params_8 = study_8.best_params

modelo_8 = xgb.XGBClassifier(**melhores_params_8)
modelo_8.fit(X_treino_sinal_estat_global_post, y_treino)

pred_8 = modelo_8.predict(X_teste_sinal_estat_global_post)
prob_8 = modelo_8.predict_proba(X_teste_sinal_estat_global_post)[:, 1]

atualizar_excel_metricas(
    arquivo="resultados_cenarios.xlsx",
    cenario_id=8,
    descricao="Sinal + Estatísticas Globais e Depois",
    y_true=y_teste,
    y_pred=pred_8,
    y_proba=prob_8
)

atualizar_parquet_predicoes(
    arquivo="predicoes_detalhadas.parquet",
    cenario_id=8,
    y_true=y_teste,
    y_pred=pred_8,
    y_proba=prob_8
)

Métricas do Cenário 8 salvas em: resultados_cenarios.xlsx
Probabilidades do Cenário 8 salvas em: predicoes_detalhadas.parquet


### 9 - Sinal + estatística global + antes + depois 

In [114]:
study_9= optuna.create_study(direction="maximize")
study_9.optimize(lambda trial: objective(trial, X_treino_sinal_estat_global_pre_post, y_treino), n_trials=15, show_progress_bar=True)

plot_optimization_history(study_9).show()
plot_param_importances(study_9).show()
plot_contour(study_9).show()

[I 2026-05-01 17:02:58,541] A new study created in memory with name: no-name-22c0c697-4abd-453e-98cb-b59457118d98
Best trial: 0. Best value: 0.311247:   7%|▋         | 1/15 [01:37<22:48, 97.73s/it]

[I 2026-05-01 17:04:36,269] Trial 0 finished with value: 0.3112466808921335 and parameters: {'learning_rate': 0.026590082732746427, 'max_depth': 11, 'n_estimators': 123}. Best is trial 0 with value: 0.3112466808921335.


Best trial: 1. Best value: 0.350407:  13%|█▎        | 2/15 [04:18<29:11, 134.70s/it]

[I 2026-05-01 17:07:16,845] Trial 1 finished with value: 0.3504067852014118 and parameters: {'learning_rate': 0.044713501796579125, 'max_depth': 13, 'n_estimators': 259}. Best is trial 1 with value: 0.3504067852014118.


Best trial: 1. Best value: 0.350407:  20%|██        | 3/15 [05:49<22:56, 114.74s/it]

...

[I 2026-05-01 17:43:06,268] Trial 12 finished with value: 0.23503084410002667 and parameters: {'learning_rate': 0.008707101717849558, 'max_depth': 263, 'n_estimators': 59}. Best is trial 7 with value: 0.39703015418438825.


Best trial: 7. Best value: 0.39703:  93%|█████████▎| 14/15 [42:57<03:06, 186.01s/it]

[I 2026-05-01 17:45:55,734] Trial 13 finished with value: 0.37901850965123707 and parameters: {'learning_rate': 0.0515093150853241, 'max_depth': 119, 'n_estimators': 229}. Best is trial 7 with value: 0.39703015418438825.


Best trial: 7. Best value: 0.39703: 100%|██████████| 15/15 [44:57<00:00, 179.83s/it]

[I 2026-05-01 17:47:56,060] Trial 14 finished with value: 0.3692943766023594 and parameters: {'learning_rate': 0.09614650060033926, 'max_depth': 251, 'n_estimators': 255}. Best is trial 7 with value: 0.39703015418438825.


In [115]:
melhores_params_9 = study_9.best_params

modelo_9 = xgb.XGBClassifier(**melhores_params_9)
modelo_9.fit(X_treino_sinal_estat_global_pre_post, y_treino)

pred_9 = modelo_9.predict(X_teste_sinal_estat_global_pre_post)
prob_9 = modelo_9.predict_proba(X_teste_sinal_estat_global_pre_post)[:, 1]

atualizar_excel_metricas(
    arquivo="resultados_cenarios.xlsx",
    cenario_id=9,
    descricao="Sinal + Estatísticas Globais, Antes e Depois",
    y_true=y_teste,
    y_pred=pred_9,
    y_proba=prob_9
)

atualizar_parquet_predicoes(
    arquivo="predicoes_detalhadas.parquet",
    cenario_id=9,
    y_true=y_teste,
    y_pred=pred_9,
    y_proba=prob_9
)

Métricas do Cenário 9 salvas em: resultados_cenarios.xlsx
Probabilidades do Cenário 9 salvas em: predicoes_detalhadas.parquet
